### **1. Install tacoreader**

## **2. Load a TACO compliant dataset**

In [4]:
import autoroot
import tacoreader
from shapely import wkb
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pathlib

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch import LightningDataModule
from loguru import logger

from src.finetuning.dataloader import Cloud3DDataModule
from src.finetuning.transforms import GeoSatTransform
from src.finetuning.model import UNetAutoencoder


In [5]:
tacoreader.use("pandas")


tacoreader.use("pandas")                      
goes = tacoreader.load("/data/databases/CLOUD_3D/pretraining/tacos/finetune/goes/")
himawari = tacoreader.load("/data/databases/CLOUD_3D/pretraining/tacos/finetune/himawari/")
msg = tacoreader.load("/data/databases/CLOUD_3D/pretraining/tacos/finetune/msg/")


# Concat
full_dataset = tacoreader.concat([goes, himawari, msg])
dataset = full_dataset.data#.to_pandas()



In [6]:
SPLITS_DICT = {
    "train": {
        "years": np.arange(2004, 2025).tolist(),
        "months": np.arange(1, 13).tolist(),
        "days": np.arange(2, 23).tolist(),
    },
    "val": {
        "years": np.arange(2004, 2025).tolist(),
        "months": np.arange(1, 13).tolist(),
        "days": np.arange(24, 27).tolist(),
    },
    "test": {
        "years": np.arange(2004, 2025).tolist(),
        "months": np.arange(1, 13).tolist(),
        "days": np.arange(28, 32).tolist(),
    },
}

def add_split_column(df: pd.DataFrame, date_col: str = "date", split_col: str = "split") -> pd.DataFrame:
    #df = df.copy()

    # ensure datetime
    df[date_col] = pd.to_datetime(df[date_col])

    y = df[date_col].dt.year
    m = df[date_col].dt.month
    d = df[date_col].dt.day

    # start with NaN / unknown
    split = np.full((len(df),), np.nan, dtype=object)
    #split = pd.Series(pd.NA, index=df.index, dtype="string")

    for name, spec in SPLITS_DICT.items():
        mask = (
            y.isin(spec["years"]) &
            m.isin(spec["months"]) &
            d.isin(spec["days"])
        )
        #split.loc[mask] = name
        split[mask] = name

    df[split_col] = split
    return df


In [7]:
dataset = add_split_column(dataset, date_col="stac:time_start", split_col="split")
dataset

                       cloud3d:cloudsat_id  cloud3d:cyclone  \
0   2018283153908_66322_CS_merged_no_flxhr            False   
1   2018283153908_66322_CS_merged_no_flxhr            False   
2   2018285151645_66351_CS_merged_no_flxhr            False   
3   2018293152524_66468_CS_merged_no_flxhr            False   
4   2018293152524_66468_CS_merged_no_flxhr            False   
..                                     ...              ...   
95  2019081152809_68704_CS_merged_no_flxhr            False   
96  2019081152809_68704_CS_merged_no_flxhr            False   
97  2019084154336_68748_CS_merged_no_flxhr            False   
98  2019084154336_68748_CS_merged_no_flxhr            False   
99  2019086152104_68777_CS_merged_no_flxhr            False   

                             cloud3d:geostationary_id  cloud3d:has_flxhr  \
0   OR_ABI-L2-MCMIPF-M3_G16_s20182831615382_e20182...              False   
1   OR_ABI-L2-MCMIPF-M3_G16_s20182831615382_e20182...              False   
2   OR_ABI-L2-M

In [8]:
transform = GeoSatTransform(patch_size=[256, 256])
dm = Cloud3DDataModule(dataset, transform)

2026-03-16 16:27:21.376 | INFO     | src.finetuning.dataloader:__init__:28 - There are 270072 files in taco dataset
2026-03-16 16:27:23.861 | INFO     | src.finetuning.dataloader:__init__:57 - MSG DataModule initialized ...
2026-03-16 16:27:23.864 | INFO     | src.finetuning.dataloader:__init__:58 - Length of train dataset: 185629
2026-03-16 16:27:23.866 | INFO     | src.finetuning.dataloader:__init__:59 - Length of test dataset: 31166
2026-03-16 16:27:23.868 | INFO     | src.finetuning.dataloader:__init__:60 - Length of val dataset: 26462


In [9]:
loader = dm.train_dataloader()

In [10]:
batch = next(iter(loader))

In [11]:
batch.keys()

dict_keys(['image', 'cloudsat', 'overpass_mask', 'satellite', 'date', 'id'])

In [ ]:
# 